# Script 3 — Treinamento dos Modelos de ML
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

| Decisão | Justificativa |
|---------|---------------|
| **9 targets** | 3 primários (DRE) + 5 balanço + 1 caixa → habilita Z''-Score prospectivo completo |
| **Split temporal** | Treino ≤2022, Teste 2023–2024 — o modelo nunca viu dados futuros |
| **GroupKFold por empresa** | Evita vazamento temporal cruzado entre empresas no CV |
| **Transformação seletiva por target** | log1p para séries positivas e arcsinh para séries negativas/mistas |
| **SMAPE como métrica principal** | Definido para negativos (Lucro Líquido pode ser negativo) |
| **Theil's U** | Prova que ML supera baseline ingênua |
| **Acurácia Direcional** | Percentual de acertos na direção (sobe/desce) |
| **Imputer no Pipeline** | Evita leakage de NaN das features YoY no CV |
| **Curvas de aprendizado** | Diagnóstico automático de overfitting/underfitting |
| **Análise de resíduos** | Valida pressupostos e detecta padrões sistemáticos |

## Etapa 0 — Dependências e configuração

In [2]:
import logging, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
logger = logging.getLogger('pipeline_modelagem')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
PASTA_LOGS = PASTA_SAIDA / 'logs'
PASTA_LOGS.mkdir(exist_ok=True)
_fh = logging.FileHandler(PASTA_LOGS / 'pipeline_modelagem.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)
# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_WF  = 5        # número de folds do Walk-Forward CV
RANDOM_STATE = 42
ANO_CORTE    = 2022     # treino ≤ 2022, teste ≥ 2023
ANOS_COVID   = {2020, 2021}   # CORREÇÃO 7: anos atípicos — flag explícita
# Targets que recebem log1p (séries positivas e assimétricas)
LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
}
# Targets com valores negativos ou mistos recebem arcsinh
ARCSINH_TARGETS = {
    'TARGET_DFC_MI_6.01',
    'TARGET_DRE_3.11',  # Lucro Líquido pode ser negativo
}
TRANSFORM_TARGETS = LOG_TARGETS | ARCSINH_TARGETS

def get_target_transform(target):
    if target in LOG_TARGETS:    return 'log1p'
    if target in ARCSINH_TARGETS: return 'arcsinh'
    return 'none'

def target_transform(y, transformacao='none'):
    y_arr = np.asarray(y, dtype=float)
    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(f'target_transform: há {n_bad} valores não finitos.')
    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            raise ValueError(
                f"target_transform(log1p): valores <= -1 encontrados. "
                f"Use 'arcsinh' para targets com negativos/mistos."
            )
        return np.log1p(y_arr)
    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)
    return y_arr.copy()

def target_inverse_transform(y_pred, transformacao='none'):
    y_arr = np.asarray(y_pred, dtype=float)
    if transformacao == 'log1p':   return np.expm1(y_arr)
    if transformacao == 'arcsinh': return np.sinh(y_arr)
    return y_arr

logger.info("Script 3 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")

2026-05-09 23:19:08 | INFO     | Script 3 iniciado | sklearn=1.8.0


✅ Dependências carregadas


## Etapa 1 — Carregamento e split temporal

In [3]:
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')
with open(PASTA_SAIDA / 'features.pkl',      'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',       'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',          'rb') as f: KPIS          = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f: GRUPOS_TREINO = pickle.load(f)
# ── Carregar splits do Script 2 ───────────────────────────────────────────
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste  = PASTA_SAIDA / 'teste.parquet'
if cam_treino.exists() and cam_teste.exists():
    treino = pd.read_parquet(cam_treino)
    teste  = pd.read_parquet(cam_teste)
    logger.info("Splits carregados dos parquets do Script 2")
else:
    logger.warning("treino.parquet não encontrado — recalculando split temporal")
    if 'split' not in dataset.columns:
        dataset['split'] = np.where(
            dataset['ANO'].astype(float) <= ANO_CORTE, 'treino',
            np.where(dataset['ANO'].astype(float) <= 2024, 'teste', 'prospectivo')
        )
    treino = dataset[dataset['split'] == 'treino'].reset_index(drop=True)
    teste  = dataset[dataset['split'] == 'teste'].reset_index(drop=True)
    GRUPOS_TREINO = treino['CNPJ_CIA'].values
    treino.to_parquet(cam_treino, index=False)
    teste.to_parquet(cam_teste,   index=False)
    with open(PASTA_SAIDA / 'grupos_treino.pkl', 'wb') as f:
        pickle.dump(GRUPOS_TREINO, f)
assert len(treino) > 0, "Treino vazio — verifique o Script 2"
assert len(teste)  > 0, "Teste vazio — verifique o Script 2"
# ── Verificação anti-leakage prospectivo ─────────────────────────────────
anos_treino = set(treino['ANO'].dropna().astype(int).unique())
anos_teste  = set(teste['ANO'].dropna().astype(int).unique())
anos_prosp  = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error("Anos prospectivos vazaram para treino/teste: %s", anos_prosp)
else:
    logger.info("Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)")
# ── CORREÇÃO 3: diagnóstico de features temporais (lags/yoy/roll) ─────────
colunas_lag = [f for f in FEATURES if any(
    s in f for s in ['_lag', '_roll', '_diff1', '_growth1', '_yoy']
)]
logger.info("Features temporais em FEATURES: %d/%d", len(colunas_lag), len(FEATURES))
print(f"  Features temporais (lags/yoy/roll): {len(colunas_lag)} de {len(FEATURES)}")
if len(colunas_lag) == 0:
    logger.warning(
        "ATENÇÃO: nenhuma feature temporal encontrada em FEATURES. "
        "Verificar seleção de features no Script 2 (RFE pode ter excluído os lags)."
    )
# ── CORREÇÃO 7: flag_covid adicionada ao treino e teste se ausente ─────────
for df_name, df in [('treino', treino), ('teste', teste)]:
    if 'flag_covid' not in df.columns:
        df['flag_covid'] = df['ANO'].isin(ANOS_COVID).astype(float)
        logger.info("flag_covid adicionada ao %s", df_name)
if 'flag_covid' not in FEATURES:
    FEATURES = list(FEATURES) + ['flag_covid']
    logger.info("flag_covid adicionada à lista FEATURES")
logger.info("Treino: %d obs (≤%d) | Teste: %d obs (%d–2024)",
            len(treino), ANO_CORTE, len(teste), ANO_CORTE + 1)
print(f"\\n{'='*65}")
print(f"  Split temporal — carregado do Script 2")
print(f"{'='*65}")
print(f"  Treino (≤{ANO_CORTE}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_treino)[:3]}...{sorted(anos_treino)[-1:]}")
print(f"  Teste ({ANO_CORTE+1}–2024): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_teste)}")
print(f"  Prospectivo (≥2025): carregado pelo Script 5")
print(f"  Isolamento prospectivo: ✅")
anos_covid_no_treino = ANOS_COVID & anos_treino
print(f"  Anos COVID no treino: {sorted(anos_covid_no_treino)} → flag_covid=1")
print(f"{'='*65}")
print(f"\\nFeatures: {len(FEATURES)} | Targets: {len(TARGETS)}")
print(f"  → Lags/YoY/Roll: {len(colunas_lag)} features temporais")
print("Targets:")
for t in TARGETS:
    if t in treino.columns:
        if t in LOG_TARGETS:       transform_flag = "📐log   "
        elif t in ARCSINH_TARGETS: transform_flag = "📐arcsinh"
        else:                      transform_flag = "         "
        n_tr = treino[t].notna().sum()
        n_te = teste[t].notna().sum() if t in teste.columns else 0
        print(f"  {transform_flag} {t:<30} treino={n_tr:,} | teste={n_te:,}")

2026-05-09 23:19:16 | INFO     | Splits carregados dos parquets do Script 2
2026-05-09 23:19:16 | INFO     | Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)
2026-05-09 23:19:16 | INFO     | Features temporais em FEATURES: 15/15
2026-05-09 23:19:16 | INFO     | flag_covid adicionada ao treino
2026-05-09 23:19:16 | INFO     | flag_covid adicionada ao teste
2026-05-09 23:19:16 | INFO     | flag_covid adicionada à lista FEATURES
2026-05-09 23:19:16 | INFO     | Treino: 713 obs (≤2022) | Teste: 168 obs (2023–2024)


  Features temporais (lags/yoy/roll): 15 de 15
\n=================================================================
  Split temporal — carregado do Script 2
  Treino (≤2022):  713 obs | DFP=181 | ITR=532 | anos [np.int64(2015), np.int64(2016), np.int64(2017)]...[np.int64(2022)]
  Teste (2023–2024):  168 obs | DFP= 24 | ITR=144 | anos [np.int64(2023), np.int64(2024)]
  Prospectivo (≥2025): carregado pelo Script 5
  Isolamento prospectivo: ✅
  Anos COVID no treino: [2020, 2021] → flag_covid=1
\nFeatures: 16 | Targets: 9
  → Lags/YoY/Roll: 15 features temporais
Targets:
  📐log    TARGET_DRE_3.01                treino=713 | teste=168
  📐arcsinh TARGET_DRE_3.11                treino=713 | teste=168
  📐log    TARGET_EBITDA                  treino=713 | teste=168
  📐log    TARGET_BPA_1                   treino=713 | teste=168
  📐log    TARGET_BPA_1.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.03                t

## Etapa 2 — Métricas, pesos por empresa e baseline ingênua

 Nesta etapa, as métricas passam a ser calculadas em nível de painel:
 - por empresa, respeitando a ordem temporal;
 - com agregação macro entre empresas;
 - e também com métrica pooled, para diagnóstico complementar.

 A baseline continua sendo a persistência do último valor observado da própria empresa.

In [4]:
def smape(y_true, y_pred):
    """SMAPE definido para qualquer sinal."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else np.nan


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def r2_seguro(y_true, y_pred):
    """R² só é válido com pelo menos 2 pontos e variância não nula."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.isclose(np.var(y_true), 0.0):
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan


def theil_u(y_true, y_pred):
    """
    Theil U calculado sobre uma série ordenada.
    U < 1 → melhor que persistência ingênua.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan

    erro_modelo = np.sqrt(np.mean((y_true[1:] - y_pred[1:]) ** 2))
    erro_baseline = np.sqrt(np.mean((y_true[1:] - y_true[:-1]) ** 2))
    return float(erro_modelo / erro_baseline) if erro_baseline > 0 else np.nan


def acuracia_direcional(y_true, y_pred):
    """
    Acurácia direcional calculada dentro de uma sequência temporal única.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan

    dir_real = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    mask = dir_real != 0
    return float(np.mean(dir_real[mask] == dir_pred[mask])) if mask.sum() > 0 else np.nan

def smape_metric(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else 0.0

smape_scorer = make_scorer(smape_metric, greater_is_better=False)


def calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='DT_REFER',
                             y_true_col='y_true', y_pred_col='y_pred'):
    """
    Calcula métricas por empresa e agrega de duas formas:
    - macro_empresa: média simples entre empresas
    - pooled: sobre todas as linhas juntas
    """
    cols = [group_col, time_col, y_true_col, y_pred_col]
    cols = [c for c in cols if c in df_eval.columns]
    df = df_eval[cols].dropna().copy()

    if df.empty:
        return {
            'n_obs_validas': 0,
            'n_empresas_validas': 0,
            'RMSE_pooled': np.nan, 'MAE_pooled': np.nan, 'SMAPE_pooled': np.nan,
            'R2_pooled': np.nan,
            'RMSE_macro_empresa': np.nan, 'MAE_macro_empresa': np.nan,
            'SMAPE_macro_empresa': np.nan, 'R2_macro_empresa': np.nan,
            'TheilU_macro_empresa': np.nan, 'DA_macro_empresa': np.nan,
        }

    if time_col in df.columns:
        df = df.sort_values([group_col, time_col], kind='mergesort')
    else:
        df = df.sort_values([group_col], kind='mergesort')

    yt_all = df[y_true_col].to_numpy(dtype=float)
    yp_all = df[y_pred_col].to_numpy(dtype=float)

    rows = []
    for emp, g in df.groupby(group_col, sort=False):
        yt = g[y_true_col].to_numpy(dtype=float)
        yp = g[y_pred_col].to_numpy(dtype=float)
        rows.append({
            'empresa': emp,
            'n_obs': len(g),
            'RMSE': rmse(yt, yp),
            'MAE': float(mean_absolute_error(yt, yp)),
            'SMAPE': smape(yt, yp),
            'R2': r2_seguro(yt, yp),
            'TheilU': theil_u(yt, yp),
            'DA': acuracia_direcional(yt, yp),
        })

    per_emp = pd.DataFrame(rows)

    return {
        'n_obs_validas': int(len(df)),
        'n_empresas_validas': int(len(per_emp)),

        'RMSE_pooled': rmse(yt_all, yp_all),
        'MAE_pooled': float(mean_absolute_error(yt_all, yp_all)),
        'SMAPE_pooled': smape(yt_all, yp_all),
        'R2_pooled': r2_seguro(yt_all, yp_all),

        'RMSE_macro_empresa': float(per_emp['RMSE'].mean()) if not per_emp.empty else np.nan,
        'MAE_macro_empresa': float(per_emp['MAE'].mean()) if not per_emp.empty else np.nan,
        'SMAPE_macro_empresa': float(per_emp['SMAPE'].mean()) if not per_emp.empty else np.nan,
        'R2_macro_empresa': float(per_emp['R2'].mean()) if not per_emp.empty else np.nan,
        'TheilU_macro_empresa': float(per_emp['TheilU'].mean()) if not per_emp.empty else np.nan,
        'DA_macro_empresa': float(per_emp['DA'].mean()) if not per_emp.empty else np.nan,
    }


def calcular_pesos_amostra(df, group_col='CNPJ_CIA'):
    """
    Peso inverso por empresa: cada companhia contribui aproximadamente com o mesmo peso
    no treino, independentemente do número de observações que possui.
    """
    if group_col not in df.columns:
        return np.ones(len(df), dtype=float)

    contagens = df[group_col].value_counts()
    pesos = 1.0 / df[group_col].map(contagens).astype(float)
    pesos = pesos / pesos.mean()
    return pesos.to_numpy(dtype=float)


def calcular_baseline(treino_df, teste_df, target):
    """
    Baseline ingênua por empresa: persistência do último valor observado
    da própria companhia, respeitando ordem temporal.
    """
    if target not in treino_df.columns or target not in teste_df.columns:
        return {}
    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns:
        return {}

    candidatos_tempo = [
        'DT_REFER', 'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next((c for c in candidatos_tempo if c in treino_df.columns and c in teste_df.columns), None)

    cols_ordenacao = ['CNPJ_CIA']
    if time_col is not None:
        cols_ordenacao.append(time_col)

    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original'] = np.arange(len(teste_tmp))

    cols_select = list(dict.fromkeys(cols_ordenacao + ['_ordem_original', target]))

    base = pd.concat([
        treino_tmp[cols_select].assign(__split='treino'),
        teste_tmp[cols_select].assign(__split='teste'),
    ], ignore_index=True)

    base = base.sort_values(cols_ordenacao + ['_ordem_original'], kind='mergesort').reset_index(drop=True)

    base['baseline_prev'] = base.groupby('CNPJ_CIA')[target].transform(lambda s: s.ffill().shift(1))

    mask_teste = base['__split'] == 'teste'
    mask_valido = mask_teste & base[target].notna() & base['baseline_prev'].notna()

    if mask_valido.sum() == 0:
        return {}

    df_eval = base.loc[mask_valido, ['CNPJ_CIA', '_ordem_original', target, 'baseline_prev']].copy()
    df_eval = df_eval.rename(columns={target: 'y_true', 'baseline_prev': 'y_pred'})

    m = calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='_ordem_original')
    cobertura = float(mask_valido.sum() / max(1, int(mask_teste.sum())))

    m['Cobertura_baseline'] = cobertura
    m['TimeCol_baseline'] = time_col if time_col is not None else ''
    return m


baselines = {}
print("=== Baseline Ingênua por empresa (persistência) ===")
print(f"  {'Target':<30} {'RMSEm':>14} {'SMAPEm':>8} {'DAm':>6} {'U':>7} {'Cob.':>6} {'ColTempo'}")
print(f"  {'-'*30} {'-'*14} {'-'*8} {'-'*6} {'-'*7} {'-'*6} {'-'*14}")

for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_macro_empresa']:>14,.0f} "
              f"{b['SMAPE_macro_empresa']:>8.1%} {b['DA_macro_empresa']:>6.1%} "
              f"{b['TheilU_macro_empresa']:>7.2f} {b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} {'N/A':>14} {'N/A':>8} {'N/A':>6} {'N/A':>7} {'N/A':>6}  N/A")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                  RMSEm   SMAPEm    DAm       U   Cob. ColTempo
  ------------------------------ -------------- -------- ------ ------- ------ --------------
  TARGET_DRE_3.01                     1,792,715     1.5%   0.0%    1.00 100.0%   DT_REFER
  TARGET_DRE_3.11                     2,176,789     9.4%   0.0%    1.00 100.0%   DT_REFER
  TARGET_EBITDA                         817,498     2.0%   0.0%    1.00 100.0%   DT_REFER
  TARGET_BPA_1                        4,098,542     2.1%   0.0%    1.00 100.0%   DT_REFER
  TARGET_BPA_1.01                     1,230,743     2.4%   0.0%    1.00 100.0%   DT_REFER
  TARGET_BPP_2.01                     1,180,294     2.6%   0.0%    1.00 100.0%   DT_REFER
  TARGET_BPP_2.03                     1,629,171     2.7%   0.0%    1.00 100.0%   DT_REFER
  TARGET_BPP_2                        4,098,542     2.1%   0.0%    1.00 100.0%   DT_REFER
  TARGET_DFC_MI_6.01                  1,089,07

## Etapa 3 — Walk-Forward temporal por período e pesos por empresa

 O split interno deixa de depender só de ANO e passa a usar a granularidade
 temporal disponível em DT_REFER quando ela existir. Isso evita misturar
 períodos diferentes dentro do mesmo ano.

In [5]:
def criar_folds_walkforward(df, time_col='DT_REFER', n_splits=N_SPLITS_WF, min_train_periods=2):
    """
    Cria folds de Walk-Forward com janela expansível.
    Usa a coluna temporal mais fina disponível.
    """
    if time_col not in df.columns:
        time_col = 'ANO' if 'ANO' in df.columns else None

    if time_col is None:
        logger.warning("Walk-Forward: nenhuma coluna temporal disponível.")
        return []

    serie_tempo = df[time_col]
    periodos = pd.Index(pd.unique(serie_tempo.dropna())).sort_values()

    if len(periodos) <= min_train_periods:
        logger.warning("Walk-Forward: períodos insuficientes para criar folds.")
        return []

    max_folds = len(periodos) - min_train_periods
    if n_splits > max_folds:
        n_splits = max(1, max_folds)
        logger.warning("Walk-Forward: períodos insuficientes, reduzindo para %d folds", n_splits)

    periodos_validacao = periodos[-n_splits:]
    folds = []

    for p_val in periodos_validacao:
        idx_tr = np.where(serie_tempo.values < p_val)[0]
        idx_val = np.where(serie_tempo.values == p_val)[0]
        if len(idx_tr) > 0 and len(idx_val) > 0:
            folds.append((idx_tr, idx_val))
            logger.debug("WF fold: treino < %s (%d obs) | val %s (%d obs)",
                         str(p_val), len(idx_tr), str(p_val), len(idx_val))

    logger.info("Walk-Forward CV: %d folds | períodos validação: %s",
                len(folds), [str(p) for p in periodos_validacao])
    return folds


# ── Ridge ──────────────────────────────────────────────────────────────────
est_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# ── SVR ────────────────────────────────────────────────────────────────────
est_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Random Forest ──────────────────────────────────────────────────────────
est_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300,
                                      random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}

# ── Gradient Boosting ──────────────────────────────────────────────────────
est_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (est_ridge, grade_ridge),
    "SVR":              (est_svr,   grade_svr),
    "RandomForest":     (est_rf,    grade_rf),
    "GradientBoosting": (est_gb,    grade_gb),
}

logger.info("%d algoritmos | Walk-Forward CV (n_splits=%d)", len(ALGORITMOS), N_SPLITS_WF)
print(f"✅ {len(ALGORITMOS)} algoritmos com Walk-Forward CV (n_splits={N_SPLITS_WF})")
print("✅ Pesos por empresa ativados no treino")

2026-05-09 23:20:17 | INFO     | 4 algoritmos | Walk-Forward CV (n_splits=5)


✅ 4 algoritmos com Walk-Forward CV (n_splits=5)
✅ Pesos por empresa ativados no treino



## Etapa 4 — Treinamento com Walk-Forward Nested Cross-Validation

 O treino agora:
 - usa Walk-Forward por período temporal fino;
 - pesa cada empresa de forma equilibrada;
 - calcula métricas por empresa no conjunto de validação de cada fold;
 - agrega os resultados em macro e pooled.

In [7]:
def treinar_alg(nome, estimador, grade, df_treino_completo,
                target, features, transformacao='none',
                n_splits_wf=N_SPLITS_WF, group_col='CNPJ_CIA', time_col='DT_REFER'):
    """
    Treinamento com Walk-Forward Nested CV.
    """
    cols_base = [c for c in [group_col, time_col, target] if c in df_treino_completo.columns]
    df_t = df_treino_completo[features + cols_base].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)

    if time_col not in df_t.columns:
        time_col = 'ANO' if 'ANO' in df_t.columns else time_col

    # Fix timezone issue: make DT_REFER tz-naive if it's tz-aware
    if time_col in df_t.columns and pd.api.types.is_datetime64tz_dtype(df_t[time_col]):
        df_t[time_col] = df_t[time_col].dt.tz_localize(None)

    X_full = df_t[features].values
    y_full = df_t[target].values
    y_fit_full = target_transform(y_full, transformacao)

    sample_weight_full = calcular_pesos_amostra(df_t, group_col=group_col)
    nome_step_final = list(estimador.named_steps.keys())[-1]
    fit_params_full = {f'{nome_step_final}__sample_weight': sample_weight_full}

    folds_ext = criar_folds_walkforward(df_t, time_col=time_col, n_splits=n_splits_wf)

    if len(folds_ext) == 0:
        logger.warning("%s | %s: sem folds válidos no Walk-Forward — fallback cv=3", nome, target)
        gs_fb = GridSearchCV(
            estimador, grade,
            cv=3,
            scoring=smape_scorer,
            refit=True, n_jobs=-1, verbose=0,
        )
        gs_fb.fit(X_full, y_fit_full, **fit_params_full)
        melhor = gs_fb.best_estimator_

        nan_m = {k: np.nan for k in [
            'RMSE_CV_macro_empresa', 'MAE_CV_macro_empresa', 'SMAPE_CV_macro_empresa',
            'R2_CV_macro_empresa', 'TheilU_CV_macro_empresa', 'DA_CV_macro_empresa',
            'RMSE_CV_pooled', 'MAE_CV_pooled', 'SMAPE_CV_pooled', 'R2_CV_pooled',
        ]}
        nan_m.update({
            'transformacao': transformacao,
            'log_transform': transformacao == 'log1p',
            'best_params': gs_fb.best_params_,
            'n_folds_wf': 0,
        })
        return melhor, nan_m

    rmse_macro_v, mae_macro_v, smape_macro_v, r2_macro_v, theil_macro_v, da_macro_v = [], [], [], [], [], []
    rmse_pool_v, mae_pool_v, smape_pool_v, r2_pool_v = [], [], [], []

    for tr_idx_ext, val_idx_ext in folds_ext:
        X_tr_ext = X_full[tr_idx_ext]
        y_tr_ext = y_fit_full[tr_idx_ext]
        y_orig_val = y_full[val_idx_ext]

        df_sub = df_t.iloc[tr_idx_ext].reset_index(drop=True)
        folds_int = criar_folds_walkforward(df_sub, time_col=time_col, n_splits=max(2, n_splits_wf - 1))
        cv_int = folds_int if len(folds_int) >= 2 else 3

        w_tr_ext = sample_weight_full[tr_idx_ext]
        fit_params_tr = {f'{nome_step_final}__sample_weight': w_tr_ext}

        gs = GridSearchCV(
            estimador, grade,
            cv=cv_int,
            scoring=smape_scorer,
            refit=True, n_jobs=-1, verbose=0,
        )
        gs.fit(X_tr_ext, y_tr_ext, **fit_params_tr)
        melhor_fold = gs.best_estimator_

        y_pred_raw = melhor_fold.predict(X_full[val_idx_ext])
        y_pred = target_inverse_transform(y_pred_raw, transformacao)

        df_fold_eval = df_t.iloc[val_idx_ext][[group_col, time_col]].copy()
        df_fold_eval['y_true'] = y_orig_val
        df_fold_eval['y_pred'] = y_pred

        m_fold = calcular_metricas_painel(
            df_fold_eval,
            group_col=group_col,
            time_col=time_col,
            y_true_col='y_true',
            y_pred_col='y_pred'
        )

        rmse_macro_v.append(m_fold['RMSE_macro_empresa'])
        mae_macro_v.append(m_fold['MAE_macro_empresa'])
        smape_macro_v.append(m_fold['SMAPE_macro_empresa'])
        r2_macro_v.append(m_fold['R2_macro_empresa'])
        theil_macro_v.append(m_fold['TheilU_macro_empresa'])
        da_macro_v.append(m_fold['DA_macro_empresa'])

        rmse_pool_v.append(m_fold['RMSE_pooled'])
        mae_pool_v.append(m_fold['MAE_pooled'])
        smape_pool_v.append(m_fold['SMAPE_pooled'])
        r2_pool_v.append(m_fold['R2_pooled'])

    # Fit final no conjunto completo de treino
    gs_final = GridSearchCV(
        estimador, grade,
        cv=folds_ext if len(folds_ext) >= 2 else 3,
        scoring=smape_scorer,
        refit=True, n_jobs=-1, verbose=0,
    )
    gs_final.fit(X_full, y_fit_full, **fit_params_full)
    melhor = gs_final.best_estimator_

    def _m(lst):
        return float(np.nanmean(lst))

    def _s(lst):
        return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV_macro_empresa': _m(rmse_macro_v),
        'RMSE_CV_macro_empresa_std': _s(rmse_macro_v),
        'MAE_CV_macro_empresa': _m(mae_macro_v),
        'SMAPE_CV_macro_empresa': _m(smape_macro_v),
        'SMAPE_CV_macro_empresa_std': _s(smape_macro_v),
        'R2_CV_macro_empresa': _m(r2_macro_v),
        'TheilU_CV_macro_empresa': _m(theil_macro_v),
        'DA_CV_macro_empresa': _m(da_macro_v),

        'RMSE_CV_pooled': _m(rmse_pool_v),
        'MAE_CV_pooled': _m(mae_pool_v),
        'SMAPE_CV_pooled': _m(smape_pool_v),
        'R2_CV_pooled': _m(r2_pool_v),

        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params': gs_final.best_params_,
        'n_folds_wf': len(folds_ext),
    }

    flag_theil = "✅" if metricas['TheilU_CV_macro_empresa'] < 1 else "⚠️"
    logger.info(
        "  %-20s RMSEm=%10.0f±%8.0f  SMAPEm=%5.1f%%  "
        "R2m=%5.3f  TheilU=%s%.3f  DAm=%.1f%%  transf=%s  folds=%d",
        nome,
        metricas['RMSE_CV_macro_empresa'], metricas['RMSE_CV_macro_empresa_std'],
        metricas['SMAPE_CV_macro_empresa'] * 100, metricas['R2_CV_macro_empresa'],
        flag_theil, metricas['TheilU_CV_macro_empresa'],
        metricas['DA_CV_macro_empresa'] * 100, transformacao, metricas['n_folds_wf']
    )

    print(
        f"  {flag_theil} {nome:<20} "
        f"RMSEm={metricas['RMSE_CV_macro_empresa']:>12,.0f}  "
        f"SMAPEm={metricas['SMAPE_CV_macro_empresa']:>5.1%}  "
        f"R²m={metricas['R2_CV_macro_empresa']:>6.3f}  "
        f"U={metricas['TheilU_CV_macro_empresa']:.3f}  "
        f"DAm={metricas['DA_CV_macro_empresa']:.1%}  "
        f"folds={metricas['n_folds_wf']}"
    )

    return melhor, metricas


# ── Execução ──────────────────────────────────────────────────────────────
resultados = {}

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\n{'='*72}")
    print(f"  TARGET: {target}  |  transform={transformacao}")
    if b:
        print(f"  Baseline → RMSEm={b.get('RMSE_macro_empresa', 0):,.0f}  "
              f"SMAPEm={b.get('SMAPE_macro_empresa', 0):.1%}  "
              f"R²m={b.get('R2_macro_empresa', 0):.3f}  "
              f"DAm={b.get('DA_macro_empresa', 0):.1%}  "
              f"Cob.={b.get('Cobertura_baseline', np.nan):.1%}")

    print(f"  {'Alg':<22} {'RMSEm':>14} {'SMAPEm':>8} {'R²m':>7} {'U':>7} {'DAm':>7}")
    print(f"  {'-'*22} {'-'*14} {'-'*8} {'-'*7} {'-'*7} {'-'*7}")

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade,
            df_treino_completo=treino,
            target=target,
            features=FEATURES,
            transformacao=transformacao,
            n_splits_wf=N_SPLITS_WF,
            group_col='CNPJ_CIA',
            time_col='DT_REFER',
        )
        resultados[target][nome] = (modelo, metricas)

        joblib.dump(
            {'modelo': modelo,
             'transformacao': transformacao,
             'log_transform': transformacao == 'log1p',
             'features': FEATURES},
            PASTA_SAIDA / f'modelo_{target}_{nome}.pkl'
        )

    logger.info("TARGET %s concluído", target)

print("\n✅ Treinamento concluído para todos os targets.")

2026-05-09 23:22:44 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2022-09-30 00:00:00', '2022-12-31 00:00:00']
2026-05-09 23:22:44 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2020-12-31 00:00:00', '2021-03-31 00:00:00', '2021-06-30 00:00:00', '2021-09-30 00:00:00']



  TARGET: TARGET_DRE_3.01  |  transform=log1p
  Baseline → RMSEm=1,792,715  SMAPEm=1.5%  R²m=0.409  DAm=0.0%  Cob.=100.0%
  Alg                             RMSEm   SMAPEm     R²m       U     DAm
  ---------------------- -------------- -------- ------- ------- -------


2026-05-09 23:22:50 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-03-31 00:00:00', '2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00']
2026-05-09 23:22:50 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00']
2026-05-09 23:22:50 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00']
2026-05-09 23:22:50 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2022-09-30 00:00:00']
2026-05-09 23:22:50 | INFO     |   Ridge                RMSEm= 519167549±571868624  SMAPEm= 62.1%  R2m=  nan  TheilU=⚠️nan  DAm=nan%  transf=log1p  folds=5
2026-05-09 23:22:50 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2

  ⚠️ Ridge                RMSEm= 519,167,549  SMAPEm=62.1%  R²m=   nan  U=nan  DAm=nan%  folds=5


2026-05-09 23:22:51 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-03-31 00:00:00', '2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00']
2026-05-09 23:22:51 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00']
2026-05-09 23:22:52 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00']
2026-05-09 23:22:53 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2022-09-30 00:00:00']
2026-05-09 23:22:55 | INFO     |   SVR                  RMSEm=  26755502± 8865533  SMAPEm= 26.0%  R2m=  nan  TheilU=⚠️nan  DAm=nan%  transf=log1p  folds=5
2026-05-09 23:22:55 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '20

...

  ⚠️ SVR                  RMSEm=   9,505,146  SMAPEm=56.6%  R²m=   nan  U=nan  DAm=nan%  folds=5


2026-05-09 23:59:02 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-03-31 00:00:00', '2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00']
2026-05-09 23:59:18 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00']
2026-05-09 23:59:34 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00']
2026-05-09 23:59:51 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2022-09-30 00:00:00']
2026-05-10 00:00:30 | INFO     |   RandomForest         RMSEm=   3654472± 2173139  SMAPEm= 62.9%  R2m=  nan  TheilU=⚠️nan  DAm=nan%  transf=arcsinh  folds=5
2026-05-10 00:00:30 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '

  ⚠️ RandomForest         RMSEm=   3,654,472  SMAPEm=62.9%  R²m=   nan  U=nan  DAm=nan%  folds=5


2026-05-10 00:00:56 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-03-31 00:00:00', '2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00']
2026-05-10 00:01:23 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-06-30 00:00:00', '2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00']
2026-05-10 00:01:51 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-09-30 00:00:00', '2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00']
2026-05-10 00:02:22 | INFO     | Walk-Forward CV: 4 folds | períodos validação: ['2021-12-31 00:00:00', '2022-03-31 00:00:00', '2022-06-30 00:00:00', '2022-09-30 00:00:00']
2026-05-10 00:03:27 | INFO     |   GradientBoosting     RMSEm=   4672452± 3585385  SMAPEm= 42.6%  R2m=  nan  TheilU=⚠️nan  DAm=nan%  transf=arcsinh  folds=5
2026-05-10 00:03:27 | INFO     | TARGET TARGET_DFC_MI_6.01 concluído


  ⚠️ GradientBoosting     RMSEm=   4,672,452  SMAPEm=42.6%  R²m=   nan  U=nan  DAm=nan%  folds=5

✅ Treinamento concluído para todos os targets.


 ## Etapa 5 — Avaliação no conjunto de teste hold-out

A avaliação passa a usar as mesmas métricas do treino: macro por empresa e pooled.

In [8]:
def avaliar_teste(modelo, df_eval, features, target, transformacao,
                  group_col='CNPJ_CIA', time_col='DT_REFER'):
    df = df_eval[features + [target] + [group_col] + [time_col]].copy()
    df = df[df[target].notna()].reset_index(drop=True)

    y_pred_raw = modelo.predict(df[features].values)
    y_pred = target_inverse_transform(y_pred_raw, transformacao)

    df_out = df[[group_col, time_col]].copy()
    df_out['y_true'] = df[target].values
    df_out['y_pred'] = y_pred

    m = calcular_metricas_painel(
        df_out,
        group_col=group_col,
        time_col=time_col,
        y_true_col='y_true',
        y_pred_col='y_pred'
    )

    return m


print("\n=== Avaliação no Teste Hold-out (2023–2024) ===")
metricas_teste = {}

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    metricas_teste[target] = {}
    baseline_rmse = b.get('RMSE_macro_empresa', np.inf)

    print(f"\n{target}  (baseline RMSEm={baseline_rmse:,.0f}  "
          f"DAm={b.get('DA_macro_empresa', 0):.1%}  "
          f"Cob.={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSEm':>14} {'SMAPEm':>8} {'R²m':>7} {'U':>7} {'DAm':>7} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*8} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")

    df_te = teste[FEATURES + [target, 'CNPJ_CIA', 'DT_REFER']].copy()

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, df_te, FEATURES, target, transformacao)
        metricas_teste[target][nome] = m

        bateu = m['RMSE_macro_empresa'] < baseline_rmse
        theil_ok = (m['TheilU_macro_empresa'] or 1.0) < 1.0
        flag = "✅" if bateu and theil_ok else ("🟡" if bateu else "❌")

        print(
            f"  {flag} {nome:<18} "
            f"{m['RMSE_macro_empresa']:>14,.0f} "
            f"{m['SMAPE_macro_empresa']:>8.1%} "
            f"{m['R2_macro_empresa']:>7.3f} "
            f"{m['TheilU_macro_empresa']:>7.3f} "
            f"{m['DA_macro_empresa']:>7.1%} "
            f"{'✅' if bateu else '❌':>7}"
        )

        logger.info(
            "Teste | %s | %s: RMSEm=%.0f SMAPEm=%.2f%% R2m=%.3f TheilU=%.3f DAm=%.1f%%",
            target, nome,
            m['RMSE_macro_empresa'], m['SMAPE_macro_empresa'] * 100,
            m['R2_macro_empresa'], m['TheilU_macro_empresa'], m['DA_macro_empresa'] * 100
        )

2026-05-10 00:03:47 | INFO     | Teste | TARGET_DRE_3.01 | Ridge: RMSEm=128241675 SMAPEm=58.82% R2m=-2066.919 TheilU=30.819 DAm=95.7%
2026-05-10 00:03:47 | INFO     | Teste | TARGET_DRE_3.01 | SVR: RMSEm=19862559 SMAPEm=22.57% R2m=-118.153 TheilU=8.453 DAm=91.3%



=== Avaliação no Teste Hold-out (2023–2024) ===

TARGET_DRE_3.01  (baseline RMSEm=1,792,715  DAm=0.0%  Cob.=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                 128,241,675    58.8% -2066.919  30.819   95.7%       ❌
  ❌ SVR                    19,862,559    22.6% -118.153   8.453   91.3%       ❌


2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.01 | RandomForest: RMSEm=6669219 SMAPEm=6.76% R2m=-62.406 TheilU=4.623 DAm=82.6%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.01 | GradientBoosting: RMSEm=4242420 SMAPEm=3.27% R2m=-16.123 TheilU=2.267 DAm=100.0%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.11 | Ridge: RMSEm=15095978 SMAPEm=171.65% R2m=-16196.281 TheilU=46.159 DAm=69.6%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.11 | SVR: RMSEm=4619370 SMAPEm=64.71% R2m=-843.352 TheilU=10.946 DAm=65.2%


  ❌ RandomForest            6,669,219     6.8% -62.406   4.623   82.6%       ❌
  ❌ GradientBoosting        4,242,420     3.3% -16.123   2.267  100.0%       ❌

TARGET_DRE_3.11  (baseline RMSEm=2,176,789  DAm=0.0%  Cob.=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                  15,095,978   171.6% -16196.281  46.159   69.6%       ❌
  ❌ SVR                     4,619,370    64.7% -843.352  10.946   65.2%       ❌


2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.11 | RandomForest: RMSEm=3359296 SMAPEm=59.85% R2m=-271.245 TheilU=8.227 DAm=47.8%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_DRE_3.11 | GradientBoosting: RMSEm=4517759 SMAPEm=49.77% R2m=-616.361 TheilU=12.275 DAm=47.8%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_EBITDA | Ridge: RMSEm=13961730 SMAPEm=72.14% R2m=-1292.079 TheilU=25.137 DAm=88.9%
2026-05-10 00:03:48 | INFO     | Teste | TARGET_EBITDA | SVR: RMSEm=10902944 SMAPEm=31.05% R2m=-114.401 TheilU=8.793 DAm=77.8%


...


TARGET_BPP_2  (baseline RMSEm=4,098,542  DAm=0.0%  Cob.=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                  62,890,645    63.0% -2611.868  33.662   65.2%       ❌
  ❌ SVR                    51,254,408    28.6% -352.374  14.893   52.2%       ❌


2026-05-10 00:03:49 | INFO     | Teste | TARGET_BPP_2 | RandomForest: RMSEm=12209473 SMAPEm=9.71% R2m=-567.031 TheilU=9.169 DAm=73.9%
2026-05-10 00:03:49 | INFO     | Teste | TARGET_BPP_2 | GradientBoosting: RMSEm=9322174 SMAPEm=5.50% R2m=-26.183 TheilU=3.415 DAm=82.6%
2026-05-10 00:03:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | Ridge: RMSEm=621362746 SMAPEm=142.92% R2m=-309637.686 TheilU=136.931 DAm=47.8%
2026-05-10 00:03:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | SVR: RMSEm=8810481 SMAPEm=43.77% R2m=-166.049 TheilU=10.660 DAm=52.2%
2026-05-10 00:03:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | RandomForest: RMSEm=3628272 SMAPEm=65.70% R2m=-531.146 TheilU=15.183 DAm=52.2%


  ❌ RandomForest           12,209,473     9.7% -567.031   9.169   73.9%       ❌
  ❌ GradientBoosting        9,322,174     5.5% -26.183   3.415   82.6%       ❌

TARGET_DFC_MI_6.01  (baseline RMSEm=1,089,073  DAm=0.0%  Cob.=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                 621,362,746   142.9% -309637.686 136.931   47.8%       ❌
  ❌ SVR                     8,810,481    43.8% -166.049  10.660   52.2%       ❌
  ❌ RandomForest            3,628,272    65.7% -531.146  15.183   52.2%       ❌


2026-05-10 00:03:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | GradientBoosting: RMSEm=3415452 SMAPEm=22.36% R2m=-35.960 TheilU=5.731 DAm=47.8%


  ❌ GradientBoosting        3,415,452    22.4% -35.960   5.731   47.8%       ❌


## Etapa 5B - Seleção do melhor modelo
Critério primário: menor SMAPE macro por empresa.
Desempates: menor TheilU macro e depois menor RMSE macro.

In [9]:
def escolher_melhor_modelo_cv(resultados_target):
    return min(
        resultados_target.items(),
        key=lambda item: (
            item[1][1].get('SMAPE_CV_macro_empresa', np.inf),
            item[1][1].get('TheilU_CV_macro_empresa', np.inf),
            item[1][1].get('RMSE_CV_macro_empresa', np.inf),
        )
    )[0]

melhores = {t: escolher_melhor_modelo_cv(resultados[t]) for t in TARGETS}

print("\n=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===")
for t, alg in melhores.items():
    m_cv = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    print(
        f"  {t:<35} → {alg:<20} "
        f"SMAPE_CV={m_cv['SMAPE_CV_macro_empresa']:.1%}  "
        f"TheilU_CV={m_cv['TheilU_CV_macro_empresa']:.3f}  "
        f"SMAPE_teste={m_test['SMAPE_macro_empresa']:.1%}"
    )


=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===
  TARGET_DRE_3.01                     → GradientBoosting     SMAPE_CV=3.2%  TheilU_CV=nan  SMAPE_teste=3.3%
  TARGET_DRE_3.11                     → GradientBoosting     SMAPE_CV=38.8%  TheilU_CV=nan  SMAPE_teste=49.8%
  TARGET_EBITDA                       → GradientBoosting     SMAPE_CV=11.7%  TheilU_CV=nan  SMAPE_teste=21.6%
  TARGET_BPA_1                        → GradientBoosting     SMAPE_CV=4.7%  TheilU_CV=nan  SMAPE_teste=5.5%
  TARGET_BPA_1.01                     → GradientBoosting     SMAPE_CV=9.0%  TheilU_CV=nan  SMAPE_teste=6.3%
  TARGET_BPP_2.01                     → GradientBoosting     SMAPE_CV=10.1%  TheilU_CV=nan  SMAPE_teste=8.0%
  TARGET_BPP_2.03                     → GradientBoosting     SMAPE_CV=8.6%  TheilU_CV=nan  SMAPE_teste=15.9%
  TARGET_BPP_2                        → GradientBoosting     SMAPE_CV=4.7%  TheilU_CV=nan  SMAPE_teste=5.5%
  TARGET_DFC_MI_6.01                  → GradientBoosting 

## Etapa 6 — Feature Importance

In [11]:
def extrair_importancia(modelo, features, nome_alg):
    step = [s for s, _ in modelo.steps][-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)

feature_importances = {}
print("\\n=== Feature Importance — Melhor Modelo por Target (SMAPE_CV) ===")
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5*n_t))
if n_t == 1: axes = [axes]
for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_mod, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome, 'importancias': imp.to_dict()}

    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        colors = ['#1f4e79' if v == top.values[0] else
                  '#2e75b6' if v >= top.values[0]*0.7 else '#9dc3e6'
                  for v in top.values[::-1]]
        axes[i].barh(range(len(top)), top.values[::-1], color=colors, alpha=0.9)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top.index[::-1], fontsize=9)
        smape_val = metricas_teste[target][melhor_nome]['SMAPE_macro_empresa']
        axes[i].set_title(
            f"{target.replace('TARGET_', '')} — {melhor_nome} (SMAPE_teste={smape_val:.1%})",
            fontsize=11, fontweight='bold'
        )
        axes[i].set_xlabel('Importância Relativa')
        axes[i].grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            axes[i].text(v + imp.max()*0.005, j, f'{v:.3f}', va='center', fontsize=8)

    top3 = list(imp.head(3).index)
    top3_flags = ['[LAG]' if any(s in f for s in ['_lag','_roll','_yoy']) else '' for f in top3]
    print(f"  {target}: {melhor_nome} | top3={[f+g for f,g in zip(top3,top3_flags)]}")
plt.suptitle('Feature Importance — Melhor Modelo por Target (SMAPE_CV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: feature_importance.png")

\n=== Feature Importance — Melhor Modelo por Target (SMAPE_CV) ===
  TARGET_DRE_3.01: GradientBoosting | top3=['TARGET_DRE_3.01_lag1[LAG]', 'flag_covid', 'TARGET_BPA_1.01_lag1[LAG]']
  TARGET_DRE_3.11: GradientBoosting | top3=['TARGET_DRE_3.11_lag1[LAG]', 'TARGET_DRE_3.01_diff1', 'TARGET_BPA_1.01_lag2[LAG]']
  TARGET_EBITDA: GradientBoosting | top3=['TARGET_BPA_1_lag1[LAG]', 'TARGET_BPP_2_lag1[LAG]', 'TARGET_DRE_3.01_lag1[LAG]']
  TARGET_BPA_1: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]', 'TARGET_BPA_1.01_lag1[LAG]']
  TARGET_BPA_1.01: GradientBoosting | top3=['TARGET_BPA_1.01_lag1[LAG]', 'TARGET_DRE_3.01_diff1', 'TARGET_BPA_1_lag1[LAG]']
  TARGET_BPP_2.01: GradientBoosting | top3=['TARGET_BPP_2.01_lag1[LAG]', 'TARGET_BPA_1.01_lag1[LAG]', 'TARGET_DRE_3.01_diff1']
  TARGET_BPP_2.03: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]', 'TARGET_BPP_2.03_lag2[LAG]']
  TARGET_BPP_2: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'T

## Etapa 7 — Curvas de Aprendizado

In [13]:
print("\\nGerando curvas de aprendizado...")
n_t = len(TARGETS)
fig, axes = plt.subplots(1, n_t, figsize=(7*n_t, 5))
if n_t == 1: axes = [axes]
for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_t = treino[FEATURES + [target, 'ANO']].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    X = df_t[FEATURES].values
    y = target_transform(df_t[target].values, transformacao)

    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]

    folds_lc = criar_folds_walkforward(df_t, time_col='DT_REFER', n_splits=N_SPLITS_WF)
    cv_lc = folds_lc if len(folds_lc) >= 2 else 3

    try:
        sizes, tr_sc, val_sc = learning_curve(
            melhor_mod, X, y,
            cv=cv_lc,
            scoring='r2',
            train_sizes=np.linspace(0.3, 1.0, 5),
            n_jobs=-1,
        )
        ax = axes[i]
        ax.plot(sizes, tr_sc.mean(1), 'o-', label='Treino',    color='#1f4e79', lw=2)
        ax.fill_between(sizes, tr_sc.mean(1)-tr_sc.std(1),
                        tr_sc.mean(1)+tr_sc.std(1), alpha=0.12, color='#1f4e79')
        ax.plot(sizes, val_sc.mean(1), 's--', label='Validação', color='#c0392b', lw=2)
        ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                        val_sc.mean(1)+val_sc.std(1), alpha=0.12, color='#c0392b')
        gap  = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
        diag = ('overfitting'  if gap > 0.15 else
                'underfitting' if val_sc.mean(1)[-1] < 0.3 else 'OK')
        ax.set_title(f"{target.replace('TARGET_', '')}\\n{melhor_nome} (Walk-Forward CV)",
                     fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Tamanho do treino  |  Gap={gap:.2f} → {diag}', fontsize=9)
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        logger.info("Curva %s/%s: gap=%.3f diag=%s", target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou %s/%s: %s", target, melhor_nome, e)
        axes[i].text(0.5, 0.5, 'Erro na curva\\n'+str(e)[:60],
                     ha='center', va='center', transform=axes[i].transAxes, fontsize=9)
plt.suptitle('Curvas de Aprendizado — Diagnóstico de Bias/Variância (Walk-Forward CV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: curvas_aprendizado.png")

2026-05-10 00:06:14 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2018', '2019', '2020', '2021', '2022']


\nGerando curvas de aprendizado...


2026-05-10 00:06:15 | INFO     | Curva TARGET_DRE_3.01/GradientBoosting: gap=-0.011 diag=OK
2026-05-10 00:06:15 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2018', '2019', '2020', '2021', '2022']
2026-05-10 00:06:16 | INFO     | Curva TARGET_DRE_3.11/GradientBoosting: gap=0.663 diag=overfitting
2026-05-10 00:06:16 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2018', '2019', '2020', '2021', '2022']
2026-05-10 00:06:18 | INFO     | Curva TARGET_EBITDA/GradientBoosting: gap=0.106 diag=OK
2026-05-10 00:06:18 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2018', '2019', '2020', '2021', '2022']
2026-05-10 00:06:19 | INFO     | Curva TARGET_BPA_1/GradientBoosting: gap=-0.021 diag=OK
2026-05-10 00:06:19 | INFO     | Walk-Forward CV: 5 folds | períodos validação: ['2018', '2019', '2020', '2021', '2022']
2026-05-10 00:06:20 | INFO     | Curva TARGET_BPA_1.01/GradientBoosting: gap=-0.015 diag=OK
2026-05-10 00:06:20 | INFO     | Walk-Forward CV: 5 fo

  ✅ Salvo: curvas_aprendizado.png


## Etapa 8 — Análise de Resíduos

CORREÇÃO 5: adicionada estratificação dos resíduos por setor.
Três painéis por target:
  1) Predito × Observado (colorido por setor)
  2) Resíduos × Predito  (colorido por setor + detector de funil)
  3) SMAPE por setor     (identifica setores com maior erro relativo)

In [16]:
print("\\nGerando análise de resíduos...")
cols_setor_disp = [c for c in teste.columns if c.startswith('setor_')]
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 3, figsize=(21, 5*n_t))
if n_t == 1: axes = axes.reshape(1, -1)
CORES_SETOR = ['#1f4e79','#c0392b','#27ae60','#f39c12','#8e44ad',
               '#2980b9','#e74c3c','#16a085','#d35400','#2c3e50']
for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_te = teste[FEATURES + [target] + cols_setor_disp].copy()
    df_te = df_te[df_te[target].notna()].reset_index(drop=True)
    X_te     = df_te[FEATURES].values
    y_te     = df_te[target].values
    melhor_nome = melhores[target]
    mod         = resultados[target][melhor_nome][0]
    y_pred      = target_inverse_transform(mod.predict(X_te), transformacao)
    residuos    = y_te - y_pred

    setor_labels    = None
    setores_unicos  = []
    if cols_setor_disp:
        setor_labels   = df_te[cols_setor_disp].idxmax(axis=1).str.replace('setor_', '')
        setores_unicos = sorted(setor_labels.unique())

    smape_val = smape(y_te, y_pred)
    r2_val = r2_score(y_te, y_pred)

    # ── Gráfico 1: Predito × Observado ───────────────────────────────────
    ax1 = axes[i, 0]
    if setor_labels is not None:
        for k, setor in enumerate(setores_unicos):
            mask_s = (setor_labels == setor).values
            ax1.scatter(y_pred[mask_s], y_te[mask_s], alpha=0.55, s=22,
                        color=CORES_SETOR[k % len(CORES_SETOR)],
                        label=setor, edgecolors='none')
        ax1.legend(fontsize=7, loc='upper left')
    else:
        ax1.scatter(y_pred, y_te, alpha=0.45, s=18, color='#1f4e79', edgecolors='none')

    lim_max = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    lim_min = min(np.nanmin(y_te), np.nanmin(y_pred)) * 1.05
    ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5)
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\\nPredito × Observado",
                  fontsize=10, fontweight='bold')
    ax1.text(0.05, 0.92, f'R²={r2_val:.3f}  SMAPE={smape_val:.1%}',
             transform=ax1.transAxes, fontsize=8,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # ── Gráfico 2: Resíduos × Predito ────────────────────────────────────
    ax2 = axes[i, 1]
    if setor_labels is not None:
        for k, setor in enumerate(setores_unicos):
            mask_s = (setor_labels == setor).values
            ax2.scatter(y_pred[mask_s], residuos[mask_s], alpha=0.55, s=22,
                        color=CORES_SETOR[k % len(CORES_SETOR)],
                        label=setor, edgecolors='none')
    else:
        ax2.scatter(y_pred, residuos, alpha=0.45, s=18, color='#744210', edgecolors='none')

    ax2.axhline(0,                  color='r',    lw=1.5, ls='--')
    ax2.axhline( np.std(residuos),  color='gray', lw=1,   ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos),  color='gray', lw=1,   ls=':', alpha=0.7)
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos × Predito\\nskew={pd.Series(residuos).skew():.2f}'
                  f'  σ={np.std(residuos):,.0f}', fontsize=10, fontweight='bold')

    # Detector de heteroscedasticidade (formato de funil)
    corr_funil = np.corrcoef(np.abs(y_pred), np.abs(residuos))[0, 1]
    if abs(corr_funil) > 0.4:
        ax2.text(0.05, 0.95, f'⚠️ Possível funil\\ncorr={corr_funil:.2f}',
                 transform=ax2.transAxes, fontsize=8, color='red', va='top',
                 bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

    # ── Gráfico 3: SMAPE por setor (CORREÇÃO 5) ───────────────────────────
    ax3 = axes[i, 2]
    if setor_labels is not None:
        smape_por_setor = {}
        for setor in setores_unicos:
            mask_s = (setor_labels == setor).values
            smape_por_setor[setor] = smape(y_te[mask_s], y_pred[mask_s]) if mask_s.sum() > 0 else np.nan

        setores_ord = sorted(smape_por_setor, key=lambda s: smape_por_setor[s] or 0)
        vals  = [smape_por_setor[s] for s in setores_ord]
        cores = [CORES_SETOR[setores_unicos.index(s) % len(CORES_SETOR)] for s in setores_ord]
        bars  = ax3.barh(setores_ord, vals, color=cores, alpha=0.85)
        ax3.axvline(smape_val, color='red', lw=1.5, ls='--', label=f'Média={smape_val:.1%}')
        ax3.set_xlabel('SMAPE')
        ax3.set_title('SMAPE por Setor', fontsize=10, fontweight='bold')
        ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
        ax3.legend(fontsize=8)
        for bar, val in zip(bars, vals):
            if val is not None and np.isfinite(val):
                ax3.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                         f'{val:.1%}', va='center', fontsize=8)
    else:
        ax3.text(0.5, 0.5, 'Setores não disponíveis',
                 ha='center', va='center', transform=ax3.transAxes, fontsize=10)
plt.suptitle('Análise de Resíduos — Teste 2023–2024 (estratificado por setor)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: analise_residuos.png")

\nGerando análise de resíduos...
  ✅ Salvo: analise_residuos.png


## Etapa 9 — Persistência completa

In [ ]:
rows_cv, rows_te = [], []
for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({
            'Target': target,
            'Algoritmo': alg,
            'RMSE_CV_macro_empresa': m['RMSE_CV_macro_empresa'],
            'RMSE_CV_macro_empresa_std': m.get('RMSE_CV_macro_empresa_std'),
            'MAE_CV_macro_empresa': m.get('MAE_CV_macro_empresa'),
            'SMAPE_CV_macro_empresa': m['SMAPE_CV_macro_empresa'],
            'SMAPE_CV_macro_empresa_std': m.get('SMAPE_CV_macro_empresa_std'),
            'R2_CV_macro_empresa': m['R2_CV_macro_empresa'],
            'TheilU_CV_macro_empresa': m.get('TheilU_CV_macro_empresa'),
            'DA_CV_macro_empresa': m.get('DA_CV_macro_empresa'),
            'RMSE_CV_pooled': m.get('RMSE_CV_pooled'),
            'SMAPE_CV_pooled': m.get('SMAPE_CV_pooled'),
            'R2_CV_pooled': m.get('R2_CV_pooled'),
            'n_folds_wf': m.get('n_folds_wf'),
            'transformacao': m.get('transformacao'),
            'log_transform': m['log_transform'],
            'best_params': str(m['best_params']),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target': target,
            'Algoritmo': alg,
            'RMSE_teste_macro_empresa': mt['RMSE_macro_empresa'],
            'MAE_teste_macro_empresa': mt['MAE_macro_empresa'],
            'SMAPE_teste_macro_empresa': mt['SMAPE_macro_empresa'],
            'R2_teste_macro_empresa': mt['R2_macro_empresa'],
            'TheilU_teste_macro_empresa': mt['TheilU_macro_empresa'],
            'DA_teste_macro_empresa': mt['DA_macro_empresa'],
            'RMSE_teste_pooled': mt['RMSE_pooled'],
            'SMAPE_teste_pooled': mt['SMAPE_pooled'],
            'R2_teste_pooled': mt['R2_pooled'],
            'RMSE_baseline': b.get('RMSE_macro_empresa'),
            'Bateu_baseline': mt['RMSE_macro_empresa'] < b.get('RMSE_macro_empresa', np.inf),
            'TheilU_ok': (mt.get('TheilU_macro_empresa', 1.0) or 1.0) < 1.0,
        })
df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)
df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv',    index=False)
df_te.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)
with open(PASTA_SAIDA / 'resultados_cv.pkl',       'wb') as f: pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl',      'wb') as f: pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl',           'wb') as f: pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl',    'wb') as f: pickle.dump(melhores, f)
relatorio = {
    'versao'             : 'V6_WalkForward_SMAPE',
    'ano_corte'          : ANO_CORTE,
    'n_treino'           : int(len(treino)),
    'n_teste'            : int(len(teste)),
    'n_splits_wf'        : N_SPLITS_WF,
    'scoring_cv'         : 'SMAPE (proporcional ao tamanho da empresa)',
    'algoritmos'         : list(ALGORITMOS.keys()),
    'targets'            : TARGETS,
    'log_targets'        : list(LOG_TARGETS),
    'arcsinh_targets'    : list(ARCSINH_TARGETS),
    'target_transforms'  : {t: get_target_transform(t) for t in TARGETS},
    'features'           : FEATURES,
    'n_features_lag'     : len(colunas_lag),
    'anos_covid'         : sorted(ANOS_COVID),
    'melhores'           : melhores,
    'baselines'          : {
        t: {k: float(v) for k, v in b.items()
            if isinstance(v, (int, float, np.floating))}
        for t, b in baselines.items()
    },
    'metodologia': {
        'cv_strategy'           : 'Walk-Forward (expanding window) por ano',
        'scoring_otimizacao'    : 'SMAPE — elimina viés de escala entre empresas',
        'selecao_melhor_modelo' : 'SMAPE_CV (sem consultar conjunto de teste)',
        'flag_covid'            : 'Anos 2020–2021 sinalizados explicitamente',
        'residuos'              : 'Estratificados por setor + detector de funil',
    },
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)
# ── Resumo final ──────────────────────────────────────────────────────────
print("\\n" + "═"*72)
print("  RESUMO FINAL — Script 3  (Walk-Forward CV + SMAPE scoring)")
print("═"*72)
print(f"  Treino   : {len(treino):,} obs (≤{ANO_CORTE}) | "
      f"DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste    : {len(teste):,} obs (≥{ANO_CORTE+1}) | "
      f"DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"  Modelos  : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  CV       : Walk-Forward, {N_SPLITS_WF} folds, scoring=SMAPE")
print(f"  Features : {len(FEATURES)} ({len(colunas_lag)} temporais: lags/yoy/roll)")
print(f"  COVID    : {sorted(ANOS_COVID)} → flag_covid=1 no dataset")
print(f"  Métricas : RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional")
print()
# CORREÇÃO 4: texto explica que seleção é por SMAPE_CV, avaliação é por métricas de teste
print("  Melhores (seleção por SMAPE_CV | avaliação por métricas de teste):")
print(f"  {'Target':<35} {'Algoritmo':<20} {'SMAPE_CV':>9} "
      f"{'SMAPE_te':>9} {'R²_te':>7} {'TheilU':>7} {'Bateu?':>7}")
print(f"  {'-'*35} {'-'*20} {'-'*9} {'-'*9} {'-'*7} {'-'*7} {'-'*7}")
for t, alg in melhores.items():
    m_cv   = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    b      = baselines.get(t, {})
    bateu  = m_test['RMSE_macro_empresa'] < b.get('RMSE_macro_empresa', np.inf)
    print(f"  {t:<35} {alg:<20} "
          f"{m_cv['SMAPE_CV_macro_empresa']:>9.1%} "
          f"{m_test['SMAPE_macro_empresa']:>9.1%} "
          f"{m_test['R2_macro_empresa']:>7.3f} "
          f"{m_test['TheilU_macro_empresa']:>7.3f} "
          f"{'✅' if bateu else '❌':>7}")
print("═"*72)
print("  ✅ Pronto para Script 4 (Avaliação + Z'')")
print("═"*72)

\n════════════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 3 V6 (Walk-Forward CV + SMAPE scoring)
════════════════════════════════════════════════════════════════════════
  Treino   : 713 obs (≤2022) | DFP=181 | ITR=532
  Teste    : 168 obs (≥2023) | DFP=24  | ITR=144
  Modelos  : 36 (9 targets × 4 algoritmos)
  CV       : Walk-Forward, 5 folds, scoring=SMAPE
  Features : 16 (15 temporais: lags/yoy/roll)
  COVID    : [2020, 2021] → flag_covid=1 no dataset
  Métricas : RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional

  Melhores (seleção por SMAPE_CV | avaliação por métricas de teste):
  Target                              Algoritmo             SMAPE_CV  SMAPE_te   R²_te  TheilU  Bateu?
  ----------------------------------- -------------------- --------- --------- ------- ------- -------
  TARGET_DRE_3.01                     GradientBoosting          3.2%      3.3% -16.123   2.267       ❌
  TARGET_DRE_3.11                     GradientBoosting 